Builds figure 5c. Heatmap of ARD residues that interact with histone residues

In [ ]:
import pandas as pd
import altair as alt
from scipy import stats
import numpy as np
from natsort import natsorted

In [ ]:
scores = './Data/final_tables/supplementary_file_1_BARD1_SGE_final_table.xlsx'#SGE datafile
alt.data_transformers.disable_max_rows()

In [ ]:
def process_data(data_path):
    df = pd.read_excel(data_path)

    df = df.loc[(df['var_type'] == 'snv') & (df['amino_acid_change'] != '---')]
    df = df.loc[~df['consequence'].isin(['stop_gained', 'synonymous_variant'])]
    df['AApos'] = df['amino_acid_change'].transform(lambda x: int(x[1:-1]))
    df['oAA'] = df['amino_acid_change'].transform(lambda x: x[0])
    df['nAA'] = df['amino_acid_change'].transform(lambda x: x[-1])

    df['aa_plot'] = df['oAA'] + df['AApos'].astype(str)
    
    ard_resis = [429, 458, 467, 462, 470, 500]

    df = df.loc[df['AApos'].isin(ard_resis)]

    aa_properties = {
            # Positive charge (basic)
            'R': 'Positive',  # Arginine
            'K': 'Positive',  # Lysine
            'H': 'Positive',  # Histidine (can be positive at physiological pH)
            
            # Negative charge (acidic)
            'D': 'Negative',  # Aspartic acid
            'E': 'Negative',  # Glutamic acid
            
            # Polar uncharged
            'S': 'Polar',     # Serine
            'T': 'Polar',     # Threonine
            'N': 'Polar',     # Asparagine
            'Q': 'Polar',     # Glutamine
            'Y': 'Polar',     # Tyrosine
            'C': 'Polar',     # Cysteine (can form disulfide bonds)
            
            # Hydrophobic (nonpolar)
            'A': 'Hydrophobic',  # Alanine
            'V': 'Hydrophobic',  # Valine
            'I': 'Hydrophobic',  # Isoleucine
            'L': 'Hydrophobic',  # Leucine
            'M': 'Hydrophobic',  # Methionine
            'F': 'Hydrophobic',  # Phenylalanine
            'W': 'Hydrophobic',  # Tryptophan
            
            # Special cases
            'G': 'Special',   # Glycine (smallest, flexible)
            'P': 'Special',   # Proline (helix breaker, rigid)
            }

    df['type_change'] = df['nAA'].map(aa_properties)

    return df

In [ ]:
def make_map(df):

    color_domain = [-0.2, 0]
    sort_order = sorted(list(set(df['aa_plot'].tolist())), key = lambda x: int(x[1:]))

    to_plot = df.groupby(['aa_plot', 'type_change']).agg({
        'score': 'median'}
                                                        ).reset_index()


    plot = alt.Chart(to_plot).mark_rect().encode(
                x = alt.X('aa_plot:O',
                          title = '',
                          sort = sort_order,
                          axis = alt.Axis(
                              labelAngle = 0,
                              ticks = False,
                              labelFontSize = 18
                          )
                         ),
                y = alt.Y('type_change:N',
                          title = '',
                          axis = alt.Axis(
                                          labelFontSize = 18
                                         )
                         ),
                color = alt.Color('score:Q',
                                 scale = alt.Scale(
                                     scheme = 'bluepurple',
                                     domain = color_domain,
                                     clamp = True,
                                     reverse = True
                                 ),
           
                                legend = alt.Legend(
                                      title = 'Fitness Score',
                                      titleFontSize = 18,
                                      labelFontSize = 16,
                                      gradientLength = 125
                                  )
                                 ),
                tooltip = ['score']
            ).properties(
                width = 450,
                height = 200,
                title = alt.TitleParams(text = 'BARD1 ARD & H4 N-Term. Tail Interaction',
                                        fontSize = 20
                                       )
            )

    plot.display()
    return plot

In [ ]:
def main(save = True):
    df = process_data(scores)
    heatmap = make_map(df)

    if save:
        heatmap.save('/Users/ivan/Desktop/BARD1_draft_figs/fig_6c_ard_heatmap.svg')


In [ ]:
main(save = False)